In [ ]:
# =============================================================================
# PROJE ADI: FİNANSMETRE (Gelişmiş Bütçe Takip Sistemi)
# GELİŞTİREN: [Damla Al] - [2433212163]
# =============================================================================

import sqlite3      # Veri tabanı işlemleri için.
import pandas as pd # Veri analizi ve Excel raporu oluşturmak için.
import tkinter as tk # Arayüzü (Pencere, Buton, Menü) tasarlayan mimar.
from tkinter import messagebox, ttk, filedialog # Uyarılar ve tablolar için ek araçlar.
from datetime import datetime # İşlem anındaki saat ve tarihi almak için.
import requests     # İnternetten canlı döviz kuru çekmek için.
import os           # Dosya ve klasör işlemleri için.

# --- AYARLAR VE SABİTLER ---
DB_NAME = 'finansmetre.db' # Verilerin kaydolacağı dosyanın ismi.
GIRIS_SIFRESI = "1234"     # Uygulama açılış şifresi.
BUTCE_LIMITI = 15000       # Kullanıcıyı uyaracağımız harcama sınırı.

# KATEGORİ LİSTESİ (Kullanıcı bunları açılır menüden seçecek)
KATEGORILER = ["Market", "Kira", "Maaş", "Fatura", "Eğlence", "Ulaşım", "Sağlık", "Diğer"]

# --- GÖRSEL TASARIM (RENKLER) ---
RENK_BG = "#0F172A"       # Ana arka plan koyu lacivert.
RENK_KART = "#1E293B"     # Panellerin bir tık daha açık gri tonu.
RENK_YESIL = "#10B981"    # Gelirler ve Kaydet butonu rengi.
RENK_KIRMIZI = "#EF4444"  # Giderler ve Sil butonu rengi.
RENK_MAVI = "#3B82F6"     # Vurgu ve Excel butonu rengi.
METIN_ANA = "#F8FAFC"     # Ana beyaz metin.
METIN_ALT = "#94A3B8"     # Yardımcı gri metin.

# 1. BÖLÜM: CANLI DÖVİZ KURU (API BAĞLANTISI)
def kur_bilgisini_cek():
    """Hocam, burada requests kütüphanesi ile dış dünyaya (API) bağlanıyoruz."""
    try:
        url = "https://api.exchangerate-api.com/v4/latest/USD" # Ücretsiz kur veri adresi.
        response = requests.get(url, timeout=5)               # Adrese istek atıyoruz.
        return response.json()['rates']['TRY']                # Gelen veriden TRY değerini ayıklıyoruz.
    except:
        return 44.50 # İnternet yoksa hata vermemesi için yedek kur.

# 2. BÖLÜM: VERİ TABANI HAZIRLIĞI (SQLite)
def veri_tabani_hazirla():
    """Programın verilerini yazacağı dijital defteri (Tabloyu) hazırlar."""
    baglanti = sqlite3.connect(DB_NAME) # Dosyaya bağlan.
    imlec = baglanti.cursor()           # İşlem yapacak kalemi eline al.
    # SQL DDL Sorgusu: Tabloyu ve sütunları (ID, Tarih, Kategori, vb.) tanımla.
    imlec.execute('''CREATE TABLE IF NOT EXISTS islemler 
                      (id INTEGER PRIMARY KEY AUTOINCREMENT, tarih TEXT, kategori TEXT, 
                       aciklama TEXT, miktar REAL, tur TEXT, dolar_karsiligi REAL)''')
    baglanti.commit() # Onayla.
    baglanti.close()  # Kapat.

# --- ANA UYGULAMA SINIFI ---
class FinansApp:
    def __init__(self, pencere):
        """Uygulama başladığında ilk çalışan ayarlar bölümü."""
        self.pencere = pencere
        self.pencere.title("FinansMetre")
        self.pencere.geometry("900x900")
        self.pencere.configure(bg=RENK_BG)
        
        self.guncel_kur = kur_bilgisini_cek() # Açılışta güncel kuru al.
        veri_tabani_hazirla()                 # Veri tabanını kontrol et.
        
        # Ekranları içine dizeceğimiz ana kutu:
        self.ana_cerceve = tk.Frame(self.pencere, bg=RENK_BG)
        self.ana_cerceve.pack(expand=True, fill="both")
        self.giris_sayfasi() # Önce şifre ekranını göster.

    # 3. BÖLÜM: ŞİFRE VE GİRİŞ KONTROLÜ
    def giris_sayfasi(self):
        """Uygulamanın kilitli giriş paneli."""
        self.login_f = tk.Frame(self.ana_cerceve, bg=RENK_KART, padx=40, pady=40, highlightthickness=1)
        self.login_f.place(relx=0.5, rely=0.5, anchor="center")
        
        tk.Label(self.login_f, text="GÜVENLİ GİRİŞ", font=("Arial", 16, "bold"), bg=RENK_KART, fg=RENK_MAVI).pack(pady=10)
        
        # Şifre kutusu (karakterleri gizli gösterir):
        self.e_sifre = tk.Entry(self.login_f, show="*", font=("Arial", 14), justify="center", 
                                bg=RENK_BG, fg="white", insertbackground="white", relief="flat")
        self.e_sifre.pack(pady=10)
        self.e_sifre.focus_set() # İmleç burada hazır başlar.
        
        # Klavyeden Enter'a basıldığında girişi tetikle:
        self.pencere.bind('<Return>', lambda event: self.sifre_onayla())
        
        tk.Button(self.login_f, text="GİRİŞ", command=self.sifre_onayla, bg=RENK_MAVI, fg="white", font=("Arial", 10, "bold"), width=15).pack(pady=10)

    def sifre_onayla(self):
        """Yazılan şifreyi kontrol eden mantık."""
        if self.e_sifre.get() == GIRIS_SIFRESI: # Eğer şifre doğruysa:
            self.pencere.unbind('<Return>')      # Enter bağını kopar (diğer ekranda karışmasın).
            self.login_f.destroy()              # Giriş panelini yok et.
            self.dashboard_yapisini_kur()        # Ana Dashboard'u çiz.
            self.verileri_yukle()                # Verileri veri tabanından getir.
        else:
            messagebox.showerror("Hata", "Giriş Reddedildi!") # Yanlışsa uyar.

    # 4. BÖLÜM: ANA DASHBOARD TASARIMI
    def dashboard_yapisini_kur(self):
        # ÜST BİLGİ PANELİ
        ust_panel = tk.Frame(self.ana_cerceve, bg=RENK_BG, pady=10)
        ust_panel.pack(fill="x", padx=30)
        tk.Label(ust_panel, text="FINANSMETRE", font=("Arial", 22, "bold"), bg=RENK_BG, fg=METIN_ANA).pack(side="left")
        tk.Label(ust_panel, text=f"USD/TRY: {self.guncel_kur} ₺", font=("Arial", 10, "bold"), bg=RENK_BG, fg=RENK_MAVI).pack(side="right")

        # ÖZET BİLGİ KARTLARI (GELİR, GİDER, BAKİYE)
        kart_paneli = tk.Frame(self.ana_cerceve, bg=RENK_BG)
        kart_paneli.pack(fill="x", padx=30, pady=10)

        def kart_ciz(baslik, renk):
            """Hocam, bu yardımcı fonksiyon ile hızlıca bilgi kartları oluşturuyorum."""
            f = tk.Frame(kart_paneli, bg=RENK_KART, padx=20, pady=15, highlightthickness=1)
            f.pack(side="left", expand=True, fill="both", padx=5)
            tk.Label(f, text=baslik, font=("Arial", 8), bg=RENK_KART, fg=METIN_ALT).pack(anchor="w")
            lbl = tk.Label(f, text="0.00 ₺", font=("Arial", 16, "bold"), bg=RENK_KART, fg=renk)
            lbl.pack(anchor="w")
            return lbl

        self.lbl_gelir = kart_ciz("TOPLAM GELİR", RENK_YESIL)
        self.lbl_gider = kart_ciz("TOPLAM GİDER", RENK_KIRMIZI)
        self.lbl_bakiye = kart_ciz("NET DURUM", RENK_MAVI)

        # VERİ GİRİŞ FORMU (FORM ALANI)
        form_cerceve = tk.Frame(self.ana_cerceve, bg=RENK_KART, padx=20, pady=20)
        form_cerceve.pack(padx=35, fill="x", pady=15)

        grid_alanı = tk.Frame(form_cerceve, bg=RENK_KART)
        grid_alanı.pack(fill="x")
        
        # --- YENİ: KATEGORİ SEÇİMİ (Açılır Menü Mantığı) ---
        # Burada kullanıcıya yazı yazdırmıyoruz, listeden seçtiriyoruz.
        f_kat = tk.Frame(grid_alanı, bg=RENK_KART)
        f_kat.grid(row=0, column=0, padx=15, sticky="ew")
        tk.Label(f_kat, text="Kategori", bg=RENK_KART, fg=METIN_ALT, font=("Arial", 9)).pack(anchor="w")
        
        self.kat_var = tk.StringVar(value=KATEGORILER[0]) # Seçilen kategoriyi tutan değişken.
        # OptionMenu bileşeni: Listeyi alıp açılır bir kutu haline getirir.
        self.opt_kat = tk.OptionMenu(f_kat, self.kat_var, *KATEGORILER)
        self.opt_kat.config(bg=RENK_BG, fg="white", font=("Arial", 10), relief="flat", highlightthickness=1, highlightbackground="#334155")
        self.opt_kat["menu"].config(bg=RENK_KART, fg="white") # Menünün iç renklerini de ayarladım.
        self.opt_kat.pack(fill="x", pady=5)

        # DİĞER GİRİŞ KUTULARI (Açıklama ve Tutar)
        self.in_aci = self.input_stili_ver(grid_alanı, "Açıklama", 1)
        self.in_mik = self.input_stili_ver(grid_alanı, "Tutar (₺)", 2)

        # İŞLEM TÜRLERİ VE BUTONLAR
        islem_cerceve = tk.Frame(form_cerceve, bg=RENK_KART)
        islem_cerceve.pack(fill="x", pady=(20,0))
        self.islem_turu = tk.StringVar(value="Gider") # Başlangıçta 'Gider' seçili.
        tk.Radiobutton(islem_cerceve, text="GİDER", variable=self.islem_turu, value="Gider", bg=RENK_KART, fg=RENK_KIRMIZI, font=("Arial", 9, "bold"), selectcolor=RENK_BG).pack(side="left")
        tk.Radiobutton(islem_cerceve, text="GELİR", variable=self.islem_turu, value="Gelir", bg=RENK_KART, fg=RENK_YESIL, font=("Arial", 9, "bold"), selectcolor=RENK_BG).pack(side="left", padx=10)
        
        # CRUD BUTONLARI (KAYDET, EXCEL, SİL)
        tk.Button(islem_cerceve, text="KAYDET", command=self.kaydet, bg=RENK_YESIL, fg="white", font=("Arial", 10, "bold"), width=12).pack(side="right")
        tk.Button(islem_cerceve, text="EXCEL AL", command=self.excel_cikti, bg=RENK_MAVI, fg="white", font=("Arial", 10, "bold"), width=12).pack(side="right", padx=10)
        tk.Button(islem_cerceve, text="SEÇİLİ SİL", command=self.islem_sil, bg=RENK_KIRMIZI, fg="white", font=("Arial", 10, "bold"), width=12).pack(side="right", padx=10)

        # VERİ TABLOSU (Treeview)
        # Verilerin listelendiği ana ızgara yapısı.
        self.tablo = ttk.Treeview(self.ana_cerceve, columns=("id", "tarih", "kat", "mik", "tur", "usd"), show='headings')
        for sutun in ("id", "tarih", "kat", "mik", "tur", "usd"):
            self.tablo.heading(sutun, text=sutun.upper())
            self.tablo.column(sutun, anchor="center", width=100)
        self.tablo.pack(padx=35, pady=10, fill="both", expand=True)
        
        tk.Label(self.ana_cerceve, text="[Damla Al] - © 2026 - Görsel Programlama Projesi", 
                 font=("Arial", 8, "italic"), bg=RENK_BG, fg=METIN_ALT).pack(side="bottom", pady=5)

    def input_stili_ver(self, parent, text, col):
        """Input kutuları için modern odaklanma (Focus) efekti sağlar."""
        f = tk.Frame(parent, bg=RENK_KART)
        f.grid(row=0, column=col, padx=15, sticky="ew")
        tk.Label(f, text=text, bg=RENK_KART, fg=METIN_ALT, font=("Arial", 9)).pack(anchor="w")
        e = tk.Entry(f, bg=RENK_BG, fg="white", font=("Arial", 11), relief="flat", insertbackground="white", highlightthickness=1, highlightbackground="#334155")
        e.pack(fill="x", pady=5, ipady=3)
        # Odaklanınca mavi yanma efekti:
        e.bind("<FocusIn>", lambda event: e.config(highlightbackground=RENK_MAVI, highlightthickness=2))
        e.bind("<FocusOut>", lambda event: e.config(highlightbackground="#334155", highlightthickness=1))
        return e

    # 5. BÖLÜM: VERİ İŞLEME VE SQL MANTIĞI
    def verileri_yukle(self):
        """SQLite veri tabanındaki verileri okur ve Dashboard kartlarını hesaplar."""
        for satir in self.tablo.get_children(): self.tablo.delete(satir) # Tabloyu tazele.
        db = sqlite3.connect(DB_NAME)
        # Pandas ile SQL'den verileri 'DataFrame' olarak çekiyoruz.
        df = pd.read_sql_query("SELECT id, tarih, kategori, miktar, tur, dolar_karsiligi FROM islemler ORDER BY id DESC", db)
        
        # Matematiksel toplamlar:
        toplam_gelir = df[df['tur'] == 'Gelir']['miktar'].sum()
        toplam_gider = df[df['tur'] == 'Gider']['miktar'].sum()
        
        # Ekrandaki renkli kartları güncelle:
        self.lbl_gelir.config(text=f"{toplam_gelir:,.2f} ₺")
        self.lbl_gider.config(text=f"{toplam_gider:,.2f} ₺")
        self.lbl_bakiye.config(text=f"{(toplam_gelir-toplam_gider):,.2f} ₺")

        # Bütçe sınırı kontrolü:
        if toplam_gider > BUTCE_LIMITI:
            messagebox.showwarning("Bütçe Kontrolü", f"Dikkat: Belirlediğin harcama limiti aşıldı!")
        db.close()
        # Satırları tabloya ekle:
        for _, row in df.iterrows():
            self.tablo.insert("", "end", values=list(row))

    def kaydet(self):
        """Kullanıcının girdiği veriyi doğrular ve SQL 'INSERT' komutu ile DB'ye yazar."""
        try:
            mik = float(self.in_mik.get()) # Yazılan miktarı al.
            usd_karsilik = round(mik / self.guncel_kur, 2) # Dolar hesabını yap.
            zaman = datetime.now().strftime("%d/%m/%Y %H:%M") # Tarihi damgala.
            
            db = sqlite3.connect(DB_NAME)
            c = db.cursor()
            # Kategori kutusundan değil, Açılır Menü (OptionMenu) değerinden alıyoruz:
            c.execute('INSERT INTO islemler (tarih, kategori, aciklama, miktar, tur, dolar_karsiligi) VALUES (?,?,?,?,?,?)',
                      (zaman, self.kat_var.get(), self.in_aci.get(), mik, self.islem_turu.get(), usd_karsilik))
            db.commit(); db.close()
            
            self.verileri_yukle() # Dashboard'u yenile.
            # Giriş alanlarını temizle:
            for e in [self.in_mik, self.in_aci]: e.delete(0, tk.END)
            messagebox.showinfo("Başarılı", "İşlemin başarıyla kaydedildi!")
        except ValueError:
            # Sayı yerine harf girilirse burası çalışır:
            messagebox.showerror("Hata", "Miktar alanına sadece sayı girmelisin Damla!")

    def islem_sil(self):
        """Seçilen satırı 'DELETE' sorgusu ile veri tabanından kalıcı olarak siler."""
        secili = self.tablo.selection()
        if not secili:
            messagebox.showwarning("Seçim Yok", "Silmek istediğin satırı listeden seçmelisin!")
            return
        
        if messagebox.askyesno("Onay", "Bu işlem veri tabanından kalıcı olarak silinecek. Emin misin?"):
            db = sqlite3.connect(DB_NAME)
            cur = db.cursor()
            for s in secili:
                # Satırdaki benzersiz ID numarasını yakala:
                id_degeri = self.tablo.item(s, "values")[0]
                cur.execute("DELETE FROM islemler WHERE id = ?", (id_degeri,))
            db.commit(); db.close()
            self.verileri_yukle() # Ekranı tazele.
            messagebox.showinfo("Başarılı", "Kayıt silindi.")

    def excel_cikti(self):
        """Pandas motoru ile veri tabanındaki her şeyi bir Excel dosyasına dönüştürür."""
        db = sqlite3.connect(DB_NAME)
        df = pd.read_sql_query("SELECT * FROM islemler", db)
        db.close()
        if df.empty: return
        
        # Kullanıcıya kaydetme yeri sorulur:
        yol = filedialog.asksaveasfilename(defaultextension=".xlsx", filetypes=[("Excel Dosyası", "*.xlsx")])
        if yol:
            df.to_excel(yol, index=False) # Excel dosyasını fiziksel olarak oluştur.
            messagebox.showinfo("Başarılı", "Finansal raporun hazır!")

# --- PROGRAMI BAŞLATMA ---
if __name__ == "__main__":
    kok = tk.Tk()
    app = FinansApp(kok)
    kok.mainloop()